### Dataset Table Creation

In [0]:
%sql
CREATE OR REPLACE TABLE non_vnet_ws.default.employees (
    employee_id   INT PRIMARY KEY,
    employee_name VARCHAR(50),
    department    VARCHAR(30),
    salary        INT
);

INSERT INTO non_vnet_ws.default.employees (employee_id, employee_name, department, salary) VALUES
(101, 'Amit Sharma',     'IT',        95000),
(102, 'Rohit Verma',     'IT',        85000),
(103, 'Neha Singh',      'IT',        85000),
(104, 'Pooja Mehta',     'IT',        78000),
(105, 'Rahul Khanna',    'IT',        72000),

(106, 'Anjali Gupta',    'HR',        65000),
(107, 'Sneha Iyer',      'HR',        60000),
(108, 'Kunal Joshi',     'HR',        60000),
(109, 'Ravi Nair',       'HR',        55000),
(110, 'Priya Menon',     'HR',        52000),

(111, 'Arjun Rao',       'Finance',   98000),
(112, 'Vikram Patel',    'Finance',   90000),
(113, 'Nitin Agarwal',   'Finance',   88000),
(114, 'Sonal Jain',      'Finance',   82000),
(115, 'Mehul Shah',      'Finance',   75000),

(116, 'Deepak Kumar',    'Sales',     70000),
(117, 'Rakesh Mishra',   'Sales',     68000),
(118, 'Sunita Das',      'Sales',     68000),
(119, 'Pankaj Roy',      'Sales',     64000),
(120, 'Alok Banerjee',   'Sales',     60000),

(121, 'Kavita Pillai',   'Marketing', 72000),
(122, 'Suresh Reddy',    'Marketing', 70000),
(123, 'Anand Chawla',    'Marketing', 68000),
(124, 'Divya Malhotra',  'Marketing', 65000),
(125, 'Manish Kapoor',   'Marketing', 62000),

(126, 'Ritu Saxena',     'Support',   58000),
(127, 'Ashish Tiwari',   'Support',   56000),
(128, 'Monika Arora',    'Support',   56000),
(129, 'Vivek Pandey',    'Support',   52000),
(130, 'Naveen Kumar',    'Support',   50000);

In [0]:
df = spark.sql("select * from non_vnet_ws.default.employees")
df.display()

### SQL using [dense_rank]

In [0]:
%sql
SELECT *
FROM (
  SELECT *,
         DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
  FROM employees
) x
WHERE rnk = 3;

per department

In [0]:
%sql
SELECT *
FROM (
  SELECT *,
         DENSE_RANK() OVER (PARTITION BY DEPARTMENT ORDER BY salary DESC) AS rnk
  FROM non_vnet_ws.default.employees
) x
WHERE rnk = 3;

### [Rank]

In [0]:
%sql
SELECT *,
         RANK() OVER (PARTITION BY DEPARTMENT ORDER BY salary DESC) AS rnk
  FROM non_vnet_ws.default.employees

### ROW_NUMBER()

In [0]:
%sql
SELECT *,
         ROW_NUMBER() OVER (PARTITION BY DEPARTMENT ORDER BY salary DESC) AS rnk
  FROM non_vnet_ws.default.employees

# PYTHON

In [0]:
def nth_highest_distinct(nums, n):
    uniq = sorted(set(nums), reverse=True)
    return uniq[n-1] if 1 <= n <= len(uniq) else None

salaries = [90000, 80000, 80000, 70000, 60000]
print(nth_highest_distinct(salaries, 2))  # 80000
print(nth_highest_distinct(salaries, 3))  # 70000
print(nth_highest_distinct(salaries, 5))  # None (not enough distinct salaries)


# PYSPARK

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.orderBy(F.col("salary").desc())
df2 = df.withColumn("rnk", F.dense_rank().over(w))
second_highest = df2.filter(F.col("rnk") == 2).drop("rnk")


LIMIT OFFSET

In [0]:
%sql
SELECT DISTINCT salary
FROM employees
ORDER BY salary DESC
LIMIT 1 OFFSET 2;

In [0]:
%sql
SELECT MAX(salary)
FROM employees
WHERE salary < (
    SELECT MAX(salary) FROM employees
);

pyspark

In [0]:
nth = 3

salaries = (
    df.select("salary")
      .distinct()
      .orderBy("salary", ascending=False)
      .limit(nth)
)

nth_highest_salary = salaries.collect()[-1][0]
print(nth_highest_salary)